In [5]:
from google.colab import files
uploaded = files.upload()  # A file picker will appear — select your CSV

Saving KAG_conversion_data.csv to KAG_conversion_data.csv


In [6]:
import pandas as pd
df = pd.read_csv('KAG_conversion_data.csv')
print(df.shape)

(1143, 11)


In [8]:
import sqlite3

# Create a database in memory
conn = sqlite3.connect('marketing.db')

# Load your cleaned data into it
df.to_sql('ad_data', conn, if_exists='replace', index=False)

print("Table loaded successfully!")

Table loaded successfully!


In [13]:
df.to_sql('ad_data', conn, if_exists='replace', index=False)
print(df.columns.tolist())  # confirm ctr and roas are there

['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']


In [17]:
print(df.columns.tolist())

['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']


In [21]:
print(pd.read_sql("SELECT * FROM ad_data LIMIT 2", conn).columns.tolist())

['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'Impressions', 'Clicks', 'Spent', 'Total_Conversion', 'Approved_Conversion']


In [24]:
import pandas as pd
import numpy as np
import sqlite3

# Load original file
df = pd.read_csv('KAG_conversion_data.csv')

# Clean column names
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Remove duplicates
df = df.drop_duplicates()

# Create new metrics
df['ctr'] = df['clicks'] / df['impressions'].replace(0, np.nan)
df['conversion_rate'] = df['approved_conversion'] / df['clicks'].replace(0, np.nan)
df['cpa'] = df['spent'] / df['approved_conversion'].replace(0, np.nan)
df['roas'] = df['total_conversion'] / df['spent'].replace(0, np.nan)

# Confirm columns exist
print("Columns:", df.columns.tolist())

# Load into SQLite
conn = sqlite3.connect('marketing.db')
df.to_sql('ad_data', conn, if_exists='replace', index=False)

# Confirm table columns
print("SQLite columns:", pd.read_sql("SELECT * FROM ad_data LIMIT 1", conn).columns.tolist())

# Run query
result1 = pd.read_sql("SELECT age, COUNT(*) as total_ads, ROUND(SUM(spent), 2) as total_spend, SUM(approved_conversion) as total_conversions, ROUND(AVG(ctr)*100, 2) as avg_ctr_pct, ROUND(AVG(roas), 4) as avg_roas FROM ad_data GROUP BY age ORDER BY total_spend DESC", conn)

print(result1)

Columns: ['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'impressions', 'clicks', 'spent', 'total_conversion', 'approved_conversion', 'ctr', 'conversion_rate', 'cpa', 'roas']
SQLite columns: ['ad_id', 'xyz_campaign_id', 'fb_campaign_id', 'age', 'gender', 'interest', 'impressions', 'clicks', 'spent', 'total_conversion', 'approved_conversion', 'ctr', 'conversion_rate', 'cpa', 'roas']
     age  total_ads  total_spend  total_conversions  avg_ctr_pct  avg_roas
0  45-49        259     20750.67                208         0.02    0.1475
1  30-34        426     15252.40                494         0.01    0.3461
2  40-44        210     11589.73                170         0.02    0.1603
3  35-39        248     11112.43                207         0.02    0.2238


In [30]:
#Performance by gender
result2 = pd.read_sql("SELECT gender, ROUND(SUM(spent), 2) as total_spend, SUM(approved_conversion) as total_conversions, ROUND(AVG(ctr)*100, 2) as avg_ctr_pct, ROUND(AVG(cpa), 2) as avg_cpa FROM ad_data GROUP BY gender", conn)
print(result2)

  gender  total_spend  total_conversions  avg_ctr_pct  avg_cpa
0      F     34502.62                495         0.02    49.75
1      M     24202.61                584         0.01    32.64


In [29]:

# Top 10 best performing ads
result3 = pd.read_sql("SELECT ad_id, age, gender, spent, approved_conversion, roas FROM ad_data WHERE spent > 0 AND approved_conversion > 0 ORDER BY roas DESC LIMIT 10", conn)
print(result3)

     ad_id    age gender  spent  approved_conversion      roas
0   777105  45-49      M   0.18                    1  5.555555
1   776416  45-49      F   0.49                    1  2.040816
2   711623  40-44      F   1.13                    1  1.769912
3   776663  30-34      M   0.57                    1  1.754386
4   780511  30-34      F   1.37                    1  1.459854
5   708746  30-34      M   1.43                    1  1.398601
6   778626  30-34      M   0.72                    1  1.388889
7   951391  30-34      F   1.59                    2  1.257862
8  1121101  30-34      M   1.59                    1  1.257862
9   738307  35-39      F   0.86                    1  1.162791


In [33]:
#Save and download

with pd.ExcelWriter('sql_results.xlsx') as writer:
    result1.to_excel(writer, sheet_name='By_Age', index=False)
    result2.to_excel(writer, sheet_name='By_Gender', index=False)
    result3.to_excel(writer, sheet_name='Top_Ads', index=False)

df.to_csv('cleaned_ad_data.csv', index=False)

from google.colab import files
files.download('sql_results.xlsx')
files.download('cleaned_ad_data.csv')

print("All done!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All done!
